# Communications — Frequency Hopping Period Identification

Simulate bursts that hop channels at a fixed period with jitter. Detect the hop period via autocorrelation / FFT,
then show how a matching `PeriodicState` explains the spectral peak placement.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import rfft, rfftfreq
from quantum_hybrid_system import PeriodicState

rng = np.random.default_rng(2)
T = 5000
hop_period = 17  # samples
jitter = 2
timeline = np.zeros(T)
pos = 0
while pos < T:
    timeline[pos] = 1.0 + 0.1*rng.normal()
    pos += hop_period + rng.integers(-jitter, jitter+1)

plt.figure()
plt.plot(timeline[:400])
plt.title("Burst timeline (first 400 samples)")
plt.xlabel("t")
plt.ylabel("amplitude")
plt.show()

yf = np.abs(rfft(timeline))
xf = rfftfreq(T, d=1.0)
k = np.argmax(yf[1:]) + 1
freq = xf[k]
period_est = 1/freq
print("Estimated hop period ~", period_est)

n = 10
r = max(2, min(int(round(period_est)), 64))
ps = PeriodicState(num_qubits=n, period=r)
samples = ps.measure(num_shots=4000, use_qft=True)

bins = 64
hist = np.zeros(bins, dtype=int)
N = 2**n
for s in samples:
    hist[(s * bins) // N] += 1

plt.figure()
plt.bar(np.arange(bins), hist)
plt.title(f"QFT histogram for hop period r≈{r}")
plt.xlabel("coarse frequency bin")
plt.ylabel("counts")
plt.show()